In [1]:
# This is necessary to recognize the modules
import os
import sys
from decimal import Decimal
import warnings

# warnings.filterwarnings("ignore")

root_path = os.path.abspath(os.path.join(os.getcwd(), '../../..'))
sys.path.append(root_path)

In [2]:
from core.backtesting import BacktestingEngine

backtesting = BacktestingEngine(root_path=root_path, load_cached_data=True)

2025-04-26 19:58:12,939 - root - ERROR - Error writing configs: [Errno 2] No such file or directory: '/home/pascal/anaconda3/envs/quants-lab/lib/python3.10/site-packages/conf/conf_client.yml'
Traceback (most recent call last):
  File "/home/pascal/anaconda3/envs/quants-lab/lib/python3.10/site-packages/hummingbot/client/config/config_helpers.py", line 876, in save_to_yml
    with open(yml_path, "w", encoding="utf-8") as outfile:
FileNotFoundError: [Errno 2] No such file or directory: '/home/pascal/anaconda3/envs/quants-lab/lib/python3.10/site-packages/conf/conf_client.yml'


In [3]:
import datetime
from decimal import Decimal, getcontext
from controllers.directional_trading.pz_ema_ribbon_trend import PZEmaRibbonTrendControllerConfig
getcontext().prec = 4  # set desired precision

# Controller configuration
connector_name = "binance_perpetual"
trading_pair = "SOL-USDT"
interval = "30m"
backtesting_resolution = "1m"

# Don't matter
cooldown_time = 60*30 #60 * 15
take_profit = 5 # 100%, -> Disable Take profit, let the trailing do it's job
stop_loss = 5

# General
total_amount_quote: int = 1000
max_executors_per_side: int = 1


# Indicator Values
ema_1: int = 10
ema_2: int = 20
ema_3: int = 30
ema_4: int = 40

natr_length: int = 9

# Triple Barrier
time_limit: int = 10800
###


# Creating the instance of the configuration and the controller
config = PZEmaRibbonTrendControllerConfig(
    connector_name=connector_name,
    leverage=20,
    trading_pair=trading_pair,
    interval=interval,
    take_profit=Decimal(take_profit),
    stop_loss=Decimal(stop_loss),
    total_amount_quote=Decimal(total_amount_quote),
    time_limit=time_limit,
    max_executors_per_side=max_executors_per_side,
    cooldown_time=cooldown_time,
    natr_length = natr_length,
    ema_1=ema_1,
    ema_2=ema_2,
    ema_3=ema_3,
    ema_4=ema_4,
)

In [4]:
# Running the backtesting this will output a backtesting result object that has built in methods to visualize the results

start = int(datetime.datetime(2025, 3, 1).timestamp())
end = int(datetime.datetime(2025, 3, 30).timestamp())
maker_fee = Decimal(0.0002)
taker_fee = Decimal(0.0006)
# Worst case, MKT entry and Stop via MKT order
trade_cost: Decimal = Decimal(3) * Decimal(taker_fee)
#+ maker_fee

backtesting_result = await backtesting.run_backtesting(config, start, end, backtesting_resolution, trade_cost=float(trade_cost))

2025-04-26 19:58:16,939 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7c45f8b016c0>
2025-04-26 19:58:16,940 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x7c4684be8a60>, 120929.227080373)])']
connector: <aiohttp.connector.TCPConnector object at 0x7c45f8b01690>
2025-04-26 19:58:18,101 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7c45f8b41450>
2025-04-26 19:58:18,103 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.client_proto.ResponseHandler object at 0x7c45f8b30700>, 120930.389034912)])']
connector: <aiohttp.connector.TCPConnector object at 0x7c45f8b41480>
2025-04-26 19:58:18,382 - asyncio - ERROR - Unclosed client session
client_session: <aiohttp.client.ClientSession object at 0x7c45f8b40970>
2025-04-26 19:58:18,383 - asyncio - ERROR - Unclosed connector
connections: ['deque([(<aiohttp.c

In [5]:
import plotly.graph_objects as go
# from plotly.subplots import make_subplots

# Let's see what is inside the backtesting results
print(backtesting_result.get_results_summary())
fig = backtesting_result.get_backtesting_figure()
# Add EMAs
candles_df = backtesting_result.processed_data

# fast_key = f"HMA_{hma_fast}"
# slow_key = f"HMA_{hma_slow}"


# fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[fast_key],
#                          line=dict(color='#00FFFF', width=2),
#                          name='Fast HMA'))
# fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[slow_key],
#                          line=dict(color='#FFFF00', width=s2),
#                          name='Slow HMA'))
# fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[f"EMA_{ema_fast}"],
#                          line=dict(color='#FFFFFF', width=2),
#                          name='Fast EMA'))
fig.add_trace(go.Scatter(x=candles_df.index, y=candles_df[f"EMA_{ema_1}"],
                         line=dict(color='#FFFFFF', width=2),
                         name='EMA 1'))



Net PNL: $282.70 (28.27%) | Max Drawdown: $-128.60 (-12.92%)
Total Volume ($): 335000.00 | Sharpe Ratio: 0.76 | Profit Factor: 1.31
Total Executors: 183 | Accuracy Long: 0.44 | Accuracy Short: 0.51
Close Types: Take Profit: 0 | Stop Loss: 0 | Time Limit: 152 |
             Trailing Stop: 0 | Early Stop: 31

